In [0]:
import boto3, requests, zipfile, io, os
from tqdm import tqdm

# Credenciais
AWS_ACCESS_KEY = dbutils.secrets.get(scope="aws-credentials", key="access-key")
AWS_SECRET_KEY = dbutils.secrets.get(scope="aws-credentials", key="secret-key")
BUCKET_NAME    = "enem-data-lake-gblzera"
REGION         = "sa-east-1"

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

url = "https://download.inep.gov.br/microdados/microdados_enem_2022.zip"
print("[2022] Iniciando download...")

response = requests.get(url, stream=True, timeout=600)
total_size = int(response.headers.get("content-length", 0))
print(f"[2022] Tamanho: {total_size / 1e9:.2f} GB")

zip_content = io.BytesIO()
for chunk in tqdm(response.iter_content(chunk_size=8192),
                  total=total_size//8192,
                  desc="Download 2022"):
    zip_content.write(chunk)

zip_content.seek(0)
print("[2022] Extraindo e enviando para S3...")

with zipfile.ZipFile(zip_content) as z:
    for file_name in z.namelist():
        if file_name.endswith(".csv") or file_name.endswith(".CSV"):
            with z.open(file_name) as f:
                s3_key = f"bronze/enem/2022/{os.path.basename(file_name)}"
                s3.upload_fileobj(f, BUCKET_NAME, s3_key)
                print(f"  ✓ {s3_key}")

print("\n✓ Upload concluído!")

In [0]:
%pip install psycopg2-binary sqlalchemy

In [0]:
from sqlalchemy import create_engine

RDS_HOST = dbutils.secrets.get(scope="aws-credentials", key="rds-host")
RDS_USER = dbutils.secrets.get(scope="aws-credentials", key="rds-user")
RDS_PASS = dbutils.secrets.get(scope="aws-credentials", key="rds-password")

engine = create_engine(
    f"postgresql+psycopg2://{RDS_USER}:{RDS_PASS}@{RDS_HOST}:5432/enem_db"
)
print("engine rds criada")

In [0]:
import pandas as pd 
import io
import boto3

AWS_ACCESS_KEY = dbutils.secrets.get(scope="aws-credentials", key="access-key")
AWS_SECRET_KEY = dbutils.secrets.get(scope="aws-credentials", key="secret-key")
RDS_HOST       = dbutils.secrets.get(scope="aws-credentials", key="rds-host")
RDS_USER       = dbutils.secrets.get(scope="aws-credentials", key="rds-user")
RDS_PASS       = dbutils.secrets.get(scope="aws-credentials", key="rds-password")
BUCKET_NAME    = "enem-data-lake-gblzera"
REGION         = "sa-east-1"

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

print("baixando dados do S3")
obj = s3.get_object(Bucket=BUCKET_NAME, Key="bronze/enem/2022/MICRODADOS_ENEM_2022.csv")
conteudo = obj["Body"].read().decode("ISO-8859-1")
print("download concluido")

COLUNAS_RAW = [
    "NU_INSCRICAO", "NU_ANO", "TP_FAIXA_ETARIA", "TP_SEXO",
    "TP_ESTADO_CIVIL", "TP_COR_RACA", "TP_NACIONALIDADE",
    "TP_ST_CONCLUSAO", "TP_ANO_CONCLUIU", "TP_ESCOLA", "TP_ENSINO",
    "IN_TREINEIRO", "CO_MUNICIPIO_RESIDENCIA", "NO_MUNICIPIO_RESIDENCIA",
    "CO_UF_RESIDENCIA", "SG_UF_RESIDENCIA",
    "NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT",
    "NU_NOTA_REDACAO", "TP_STATUS_REDACAO",
    "Q001", "Q002", "Q003", "Q004", "Q005", "Q006", "Q007", "Q008", "Q009", "Q010"
]

print("caraga incial de dados no schema raw.enem_microdados")
chunk_num = 0
total_inserido = 0

for chunk in pd.read_csv(
    io.StringIO(conteudo),
    sep=";",
    low_memory=False,
    chunksize=50_000,
    usecols=lambda c: c in COLUNAS_RAW
):
    chunk.columns = [c.lower() for c in chunk.columns]
    chunk = chunk.astype(str).replace("nan", None)

    chunk.to_sql(
        "enem_microdados",
        engine,
        schema="raw",
        if_exists="append",
        index=False,
        method="multi",
        chunksize=1000
    )

    chunk_num += 1
    total_inserido += len(chunk)
    print(f"chunk {chunk_num} - {total_inserido:,} registros inseridos")

print(f"\ncarga RAW concluida - {total_inserido:,} registro")


Tivemos que alterar algumas colunas por conta de como os dados estao sujos na base

ALTER TABLE raw.enem_microdados
    ALTER COLUMN tp_faixa_etaria    TYPE TEXT,
    ALTER COLUMN tp_estado_civil    TYPE TEXT,
    ALTER COLUMN tp_cor_raca        TYPE TEXT,
    ALTER COLUMN tp_nacionalidade   TYPE TEXT,
    ALTER COLUMN tp_st_conclusao    TYPE TEXT,
    ALTER COLUMN tp_ano_concluiu    TYPE TEXT,
    ALTER COLUMN tp_escola          TYPE TEXT,
    ALTER COLUMN tp_ensino          TYPE TEXT,
    ALTER COLUMN in_treineiro       TYPE TEXT,
    ALTER COLUMN tp_status_redacao  TYPE TEXT,
    ALTER COLUMN q001               TYPE TEXT,
    ALTER COLUMN q002               TYPE TEXT,
    ALTER COLUMN q003               TYPE TEXT,
    ALTER COLUMN q004               TYPE TEXT,
    ALTER COLUMN q005               TYPE TEXT,
    ALTER COLUMN q006               TYPE TEXT,
    ALTER COLUMN q007               TYPE TEXT,
    ALTER COLUMN q008               TYPE TEXT,
    ALTER COLUMN q009               TYPE TEXT,
    ALTER COLUMN q010               TYPE TEXT;